In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
.appName("Schema Enforcement in Spark") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 22:06:44 INFO SparkEnv: Registering MapOutputTracker
26/04/10 22:06:44 INFO SparkEnv: Registering BlockManagerMaster
26/04/10 22:06:44 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/10 22:06:44 INFO SparkEnv: Registering OutputCommitCoordinator


In [ ]:
df = spark.read \
.format('csv') \
.option('header',"true") \
.option('inferSchema','true') \
.load('/data/customers_300mb.csv')

#  ---> will be scanning the whole data and can also lead to wrong inference , not consistent

### Struct Type

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, DataType, FloatType, StringType, BooleanType

In [4]:
schema = StructType([StructField('customer_id', IntegerType(), False),
                    StructField('name', StringType(), False),
                    StructField('city', StringType(), False),
                    StructField('state', StringType(), False),
                    StructField('country', StringType(), False),
                    StructField('registration_date', StringType(), False),
                    StructField('is_active', BooleanType(), False),
                    ])

In [5]:
df = spark.read \
.format('csv') \
.option('header',"true") \
.schema(schema) \
.load('/data/customers_150mb.csv')

In [6]:
df.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|  Kolkata|  Karnataka|  India|       2023-12-21|    false|
|          1| Customer_1|Ahmedabad|  Telangana|  India|       2023-03-07|     true|
|          2| Customer_2|Bangalore|  Karnataka|  India|       2023-01-14|    false|
|          3| Customer_3|  Chennai|  Karnataka|  India|       2023-11-16|    false|
|          4| Customer_4|    Delhi|  Telangana|  India|       2023-05-23|     true|
|          5| Customer_5|Hyderabad|Maharashtra|  India|       2023-07-10|    false|
|          6| Customer_6|     Pune|West Bengal|  India|       2023-03-04|     true|
|          7| Customer_7|Hyderabad|West Bengal|  India|       2023-08-11|    false|
|          8| Customer_8|  Chennai|West Bengal|  India|       2023-10-21|   

In [8]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: boolean (nullable = true)



### DDL String

In [11]:
!hadoop fs -ls /data/

Found 5 items
-rw-r--r--   2 root hadoop    1060750 2026-04-10 21:42 /data/customers.csv
-rw-r--r--   2 root hadoop       5488 2026-04-10 15:25 /data/customers_100.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-10 21:41 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-10 21:40 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop  236978176 2026-04-10 16:31 /data/customers_300mb.csv


In [9]:
ddl_schema = 'customer_id INT, name STRING, city STRING,state STRING, country STRING, registration_date STRING, is_active BOOLEAN'

In [14]:
df_ddl = spark.read \
.format("csv") \
.option("header", "true") \
.schema(ddl_schema) \
.load("/data/customers_100.csv")

In [15]:
df_ddl.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: boolean (nullable = true)



In [16]:
df_ddl.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2| Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3| Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
|          5| Customer_5|Hyderabad|  Karnataka|  India|       2023-07-28|    false|
|          6| Customer_6|     Pune|      Delhi|  India|       2023-08-29|    false|
|          7| Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|     true|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-06-22|   

In [17]:
spark.stop()